In [1]:
import pandas as pd
from pathlib import Path

from langchain_core.tools import tool

In [2]:
data_path = Path("../data/structured")

employees = pd.read_csv(
    data_path / "employees.csv"
)

attendance = pd.read_csv(
    data_path / "attendance.csv"
)

leave = pd.read_csv(
    data_path / "leave.csv"
)

holidays = pd.read_csv(
    data_path / "holidays.csv"
)

attendance["date"] = pd.to_datetime(
    attendance["date"]
)

leave["start_date"] = pd.to_datetime(
    leave["start_date"]
)

leave["end_date"] = pd.to_datetime(
    leave["end_date"]
)

holidays["date"] = pd.to_datetime(
    holidays["date"]
)

print("Data loaded successfully.")

Data loaded successfully.


In [4]:
@tool
def get_employee(employee_id: str) -> str:
    """
    Get information about an employee.

    Use this tool when you need employee information
    such as name, department, role, employment type,
    weekly working hours, or vacation entitlement.
    """

    result = employees[
        employees["employee_id"] == employee_id
    ]

    if result.empty:
        return f"No employee found with ID {employee_id}."

    employee = result.iloc[0]

    return (
        f"Employee ID: {employee['employee_id']}\n"
        f"Name: {employee['name']}\n"
        f"Department: {employee['department']}\n"
        f"Role: {employee['role']}\n"
        f"Employment type: {employee['employment_type']}\n"
        f"Weekly hours: {employee['weekly_hours']}\n"
        f"Vacation entitlement: {employee['vacation_days']} days\n"
        f"Office location: {employee['office_location']}"
    )

In [5]:
print(
    get_employee.invoke({
        "employee_id": "E0001"
    })
)

Employee ID: E0001
Name: Anna Müller
Department: Data & AI
Role: Data Analyst
Employment type: Full-time
Weekly hours: 40
Vacation entitlement: 30 days
Office location: Stuttgart


In [6]:
@tool
def get_attendance_summary(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Get attendance statistics for an employee
    during a specified date range.

    Returns office days, home-office days,
    business-trip days, sick days, leave days,
    and missing attendance records.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    if records.empty:
        return (
            f"No attendance records found for "
            f"{employee_id} between {start_date} and {end_date}."
        )

    office_days = (
        records["location"] == "office"
    ).sum()

    home_days = (
        records["location"] == "home"
    ).sum()

    business_trip_days = (
        records["location"] == "business_trip"
    ).sum()

    sick_days = (
        records["status"] == "sick"
    ).sum()

    leave_days = (
        records["status"] == "leave"
    ).sum()

    missing_records = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ].shape[0]

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Office days: {office_days}\n"
        f"Home-office days: {home_days}\n"
        f"Business-trip days: {business_trip_days}\n"
        f"Sick days: {sick_days}\n"
        f"Leave days: {leave_days}\n"
        f"Missing attendance records: {missing_records}"
    )

In [7]:
print(
    get_attendance_summary.invoke({
        "employee_id": "E0001",
        "start_date": "2026-08-01",
        "end_date": "2026-08-31"
    })
)

Employee: E0001
Period: 2026-08-01 to 2026-08-31
Office days: 15
Home-office days: 5
Business-trip days: 1
Sick days: 0
Leave days: 0
Missing attendance records: 3


In [8]:
@tool
def calculate_working_hours(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Calculate the total recorded working hours
    for an employee during a date range.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return (
            f"No complete attendance records found "
            f"for {employee_id}."
        )

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    total_hours = records["hours"].sum()

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Total recorded working hours: "
        f"{total_hours:.2f}"
    )

In [9]:
@tool
def find_missing_attendance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Find dates where an employee has missing
    attendance or a missing check-out.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    missing = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ]

    if missing.empty:
        return (
            f"No missing attendance records found "
            f"for {employee_id} between "
            f"{start_date} and {end_date}."
        )

    results = []

    for _, row in missing.iterrows():

        results.append(
            f"{row['date'].date()} - "
            f"{row['status']}"
        )

    return (
        f"Missing attendance for {employee_id}:\n"
        + "\n".join(results)
    )

In [10]:
@tool
def get_leave_balance(
    employee_id: str
) -> str:
    """
    Calculate the employee's vacation entitlement,
    approved vacation used, and remaining vacation.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return f"No employee found with ID {employee_id}."

    employee = employee_result.iloc[0]

    entitlement = int(
        employee["vacation_days"]
    )

    approved_vacation = leave[
        (leave["employee_id"] == employee_id) &
        (leave["type"] == "vacation") &
        (leave["status"] == "approved")
    ]

    used = int(
        approved_vacation["days"].sum()
    )

    remaining = entitlement - used

    return (
        f"Employee: {employee_id}\n"
        f"Vacation entitlement: {entitlement} days\n"
        f"Approved vacation used: {used} days\n"
        f"Remaining vacation: {remaining} days"
    )

In [11]:
print(
    get_leave_balance.invoke({
        "employee_id": "E0001"
    })
)

Employee: E0001
Vacation entitlement: 30 days
Approved vacation used: 5 days
Remaining vacation: 25 days


In [12]:
tools = [
    get_employee,
    get_attendance_summary,
    calculate_working_hours,
    find_missing_attendance,
    get_leave_balance
]

print("Available tools:")

for tool in tools:
    print("-", tool.name)

Available tools:
- get_employee
- get_attendance_summary
- calculate_working_hours
- find_missing_attendance
- get_leave_balance
